In [1]:
import pandas as pd
import numpy as np

# 1. Ingest Data Matrix
df = pd.read_csv("../../01_data/raw/dataset_raw.csv")

# 2. Handle High-Missingness Features (>50% Nulls)
null_percentages = df.isnull().mean()
columns_to_drop = null_percentages[null_percentages > 0.50].index.tolist()
df.drop(columns=columns_to_drop, inplace=True)

# 3. Handle Remaining Irregularities & Duplicates
duplicate_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Removed {duplicate_count} exact duplicate rows.")

# 4. Programmatic Data Imputation
# Categorical -> Impute with 'Unknown'
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Numeric -> Impute remaining missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# 5. Type Conversions & Feature Engineering
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month

# Feature 1: Processing Latency (Shipping Delay Days)
df['Shipping Delay'] = (df['Ship Date'] - df['Order Date']).dt.days

# Feature 2: Simulated Profit Target Vector (15% Revenue Capture)
df['Total Profit'] = df['Sales'] * 0.15

# 6. Relational Aggregations via .groupby()
region_summary = df.groupby('Region', as_index=False)['Sales'].mean()
region_summary.rename(columns={'Sales': 'Average_Sales_Per_Region'}, inplace=True)

# 7. Persist Transformed Files to Disk Paths
df.to_csv("../../01_data/processed/dataforge_cleaned.csv", index=False)
region_summary.to_csv("../../01_data/processed/region_sales_aggregated.csv", index=False)
print("Pipeline steps ran and outputs saved successfully!")

Removed 0 exact duplicate rows.
Pipeline steps ran and outputs saved successfully!


C:\Users\ksund\AppData\Local\Temp\ipykernel_15708\883418966.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns
